# Notebook 04: Layer Intervention Depth Characterization Benchmark

This notebook demonstrates how to programmatically execute the **Layer Intervention Benchmark** across multiple transformer layers using `src.benchmark.layer_benchmark_runner` with stage profiling, extended metadata, and multi-prompt batch execution.

In [ ]:
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

from src.benchmark.layer_benchmark_runner import run_layer_benchmark
from src.sae_utils import load_base_model, load_sae_for_layer

## 1. Multi-Prompt Batch Benchmark Sweep Across Layers `[2, 5, 8, 10]`

In [ ]:
prompts_dataset = [
    {"prompt": "The location of Massachusetts Institute of Technology is in", "target": "Cambridge"},
    {"prompt": "The capital of France is", "target": "Paris"}
]

def stage_callback(p_idx, total_p, l_idx, total_l, layer, prompt_text, stage_name):
    print(f"[Prompt {p_idx}/{total_p}] Layer {layer} ({l_idx}/{total_l}) | Stage: {stage_name}")

results = run_layer_benchmark(
    prompts=prompts_dataset,
    layers=[2, 5, 8, 10],
    mute_strength=0.3,
    boost_strength=0.5,
    mute_batch_size=3,
    boost_batch_size=3,
    use_safety=True,
    algorithm="hybrid",
    layer_callback=stage_callback
)

print(f"\nBenchmark completed! Saved to single master artifact: {results.get('saved_filepath')}")

## 2. Inspect Experiment Metadata

In [ ]:
print("Master Benchmark Metadata:")
print(f" - Benchmark Version: {results.get('benchmark_version')}")
print(f" - Model: {results.get('model')}")
print(f" - SAE Release: {results.get('sae_release')}")
print(f" - Hook Location: {results.get('hook_location')}")
print(f" - Safety Enabled: {results.get('safety_enabled')}")
print(f" - Selected Layers: {results.get('selected_layers')}")

## 3. Results DataFrame & Stage Profiling Breakdown (Prompt #1)

In [ ]:
prompt1_run = results["prompts"][0]
df_layers = pd.DataFrame(prompt1_run["layers"])
df_layers["probability_gain_pct"] = df_layers["probability_gain"] * 100

print(f"Results for Prompt 1: '{prompt1_run['prompt']}' -> Target: '{prompt1_run['target']}'")
display(df_layers[["layer", "clean_rank", "final_rank", "rank_improvement", "clean_probability", "final_probability", "probability_gain_pct", "runtime_ms", "success"]])

# Stage Profiling Breakdown
profile_data = [r["profile"] for r in prompt1_run["layers"]]
df_profile = pd.DataFrame(profile_data)
df_profile["layer"] = df_layers["layer"]
print("\nStage Profiling Breakdown (ms):")
display(df_profile[["layer", "sae_loading_ms", "clean_baseline_ms", "feature_selection_ms", "safety_filtering_ms", "intervention_ms", "total_layer_ms"]])

## 4. Visualize Target Probability Gain (%) vs Layer Depth

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar([f"Layer {l}" for l in df_layers["layer"]], df_layers["probability_gain_pct"], color="#4C72B0")
plt.title(f"Probability Gain (%) vs Layer Depth ('{prompt1_run['target']}')")
plt.ylabel("Probability Gain (%)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()